# Tugas 02 – Evaluasi Kinerja Model Klasifikasi Menggunakan Confusion Matrix
**Nama:** Ali Muhammad Firdaus  
**NIM:** H1D024102  

Studi kasus: klasifikasi citra hewan (Kucing, Ikan, Ayam) dengan 100 citra pengujian.

## 1. Import Library

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.options.display.float_format = '{:.2f}'.format

## 2. Membuat Confusion Matrix
Baris = kelas aktual, kolom = kelas prediksi.

In [2]:
kelas = ['Kucing', 'Ikan', 'Ayam']

cm = np.array([[28,  1,  1],
               [ 2, 25,  3],
               [ 1,  2, 37]])

df_cm = pd.DataFrame(cm, index=[f'Aktual {k}' for k in kelas],
                         columns=[f'Prediksi {k}' for k in kelas])
N = cm.sum()

print('Confusion Matrix:')
display(df_cm)
print('Total data       :', N)
print('Prediksi benar   :', np.trace(cm))
print('Prediksi salah   :', N - np.trace(cm))

Confusion Matrix:


,Prediksi Kucing,Prediksi Ikan,Prediksi Ayam
Aktual Kucing,28,1,1
Aktual Ikan,2,25,3
Aktual Ayam,1,2,37


Total data       : 100
Prediksi benar   : 90
Prediksi salah   : 10


## 3. Nilai TP, FP, FN, TN per Kelas (One-vs-Rest)

In [3]:
hasil = {}
for i, k in enumerate(kelas):
    TP = cm[i, i]
    FP = cm[:, i].sum() - TP
    FN = cm[i, :].sum() - TP
    TN = N - TP - FP - FN
    hasil[k] = dict(TP=TP, FP=FP, FN=FN, TN=TN)

df_ovr = pd.DataFrame(hasil).T
display(df_ovr)

,TP,FP,FN,TN
Kucing,28,3,2,67
Ikan,25,3,5,67
Ayam,37,4,3,56


## 4. Metrik Evaluasi per Kelas

In [4]:
def hitung_metrik(TP, FP, FN, TN):
    acc = (TP + TN) / (TP + TN + FP + FN)
    prec = TP / (TP + FP)
    rec = TP / (TP + FN)
    f1 = 2 * prec * rec / (prec + rec)
    fpr = FP / (FP + TN)
    fnr = FN / (FN + TP)
    spec = TN / (TN + FP)
    return dict(Accuracy=acc, Precision=prec, Recall=rec, F1=f1, FPR=fpr, FNR=fnr, Specificity=spec)

metrik = {k: hitung_metrik(**v) for k, v in hasil.items()}
df_metrik = pd.DataFrame(metrik).T * 100
print('Hasil evaluasi setiap kelas (%):')
display(df_metrik)

## 5. Macro Average dan Micro Average

In [5]:
macro_p = df_metrik['Precision'].mean()
macro_r = df_metrik['Recall'].mean()
macro_f1 = df_metrik['F1'].mean()

sTP = df_ovr['TP'].sum()
sFP = df_ovr['FP'].sum()
sFN = df_ovr['FN'].sum()
micro_p = sTP / (sTP + sFP) * 100
micro_r = sTP / (sTP + sFN) * 100
micro_f1 = 2 * micro_p * micro_r / (micro_p + micro_r)

df_avg = pd.DataFrame({'Precision': [macro_p, micro_p], 'Recall': [macro_r, micro_r], 'F1-Score': [macro_f1, micro_f1]}, index=['Macro Average', 'Micro Average'])
print(f'Sum TP={sTP}, Sum FP={sFP}, Sum FN={sFN}')
display(df_avg)

## 6. Visualisasi Confusion Matrix (Heatmap)

In [6]:
fig, ax = plt.subplots(figsize=(6.5, 5.2))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
            xticklabels=kelas, yticklabels=kelas,
            annot_kws={'size': 15, 'weight': 'bold'},
            linewidths=1, linecolor='white', ax=ax)
ax.set_xlabel('Kelas Prediksi', fontsize=12)
ax.set_ylabel('Kelas Aktual', fontsize=12)
ax.set_title('Confusion Matrix Klasifikasi Citra Hewan', fontsize=13)
plt.tight_layout()
plt.savefig('confusion_matrix_heatmap.png', dpi=200)
plt.show()

## 7. Verifikasi terhadap Perhitungan Manual

In [7]:
manual = {
    'Kucing': dict(Accuracy=95.00, Precision=90.32, Recall=93.33, F1=91.80, FPR=4.29, FNR=6.67, Specificity=95.71),
    'Ikan': dict(Accuracy=92.00, Precision=89.29, Recall=83.33, F1=86.21, FPR=4.29, FNR=16.67, Specificity=95.71),
    'Ayam': dict(Accuracy=93.00, Precision=90.24, Recall=92.50, F1=91.36, FPR=6.67, FNR=7.50, Specificity=93.33),
}
for k in kelas:
    for m, v in manual[k].items():
        assert round(df_metrik.loc[k, m], 2) == v, (k, m)
print('Semua nilai program SESUAI dengan perhitungan manual.')

from sklearn.metrics import precision_recall_fscore_support
y_true = np.repeat(np.arange(3), cm.sum(axis=1))
y_pred = np.concatenate([np.repeat(np.arange(3), cm[i]) for i in range(3)])
for avg in ['macro', 'micro']:
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average=avg)
    print(f'{avg:5s} -> P={p*100:.2f}  R={r*100:.2f}  F1={f*100:.2f}')

## 8. Kesimpulan
- Dari 100 citra, **90 diprediksi benar** dan **10 salah** (accuracy keseluruhan 90,00%).
- **Kucing** memiliki Precision (90,32%) dan F1-Score (91,80%) tertinggi; **Ikan** memiliki Recall terendah (83,33%) dan FNR tertinggi (16,67%).
- **Ayam** memiliki FPR tertinggi (6,67%) karena paling sering menjadi tempat salah prediksi dari kelas lain.
- Macro average: Precision 89,95%, Recall 89,72%, F1-Score 89,79%. Micro average: Precision, Recall, dan F1-Score sama-sama 90,00%.